In [1]:
# Cell 1 — CONFIG (EDIT PATHS)
import os, re, json
from pathlib import Path
import numpy as np
import pandas as pd

# ===== EDIT THESE =====
EDF_DIR_MESA = Path("C:/Users/Zubair/mesa/polysomnography/edfs")          # contains mesa-*.edf
ANNOT_DIR     = Path("C:/Users/Zubair/mesa/polysomnography/annotations-events-profusion")  # contains *-profusion.xml

OUT_ROOT      = Path("./mesa_sleepstaging_planA")
OUT_MESA      = OUT_ROOT / "npz_mesa"
MANIFEST_DIR  = OUT_ROOT / "manifests"

for p in [OUT_MESA, MANIFEST_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ===== RULES =====
PILOT_N = 10

EPOCH_SEC = 30
TARGET_FS = 125

TRIM_MINUTES_START = 20
TRIM_MINUTES_END   = 20
MIN_TST_HOURS      = 4.0

# Single-channel EEG ONLY (treat as C4–A1 equivalent)
EEG_CHANNEL_NAME = "EEG1"

print("CONFIG OK")
print("EDF_DIR_MESA:", EDF_DIR_MESA)
print("ANNOT_DIR    :", ANNOT_DIR)
print("OUT_ROOT     :", OUT_ROOT)


CONFIG OK
EDF_DIR_MESA: C:\Users\Zubair\mesa\polysomnography\edfs
ANNOT_DIR    : C:\Users\Zubair\mesa\polysomnography\annotations-events-profusion
OUT_ROOT     : mesa_sleepstaging_planA


In [2]:
# Cell 2 — Imports + MNE quiet
import mne
mne.set_log_level("WARNING")

In [3]:
# Cell 3 — Utilities: listing + ids + stage mapping
def list_edf_files(edf_dir: Path):
    return sorted(edf_dir.rglob("*.edf"))

def infer_record_id_from_edf(edf_path: Path):
    return edf_path.stem  # e.g., shhs1-200001

def normalize_keys_from_rec_id(rec_id: str):
    rec_id = str(rec_id)
    digits = "".join(re.findall(r"\d+", rec_id))
    keys = []
    keys.append(rec_id)
    keys.append(rec_id.replace("-", "_"))
    keys.append(rec_id.replace("-", ""))
    if digits:
        keys.append(digits)
        prefix = rec_id.split("-")[0] if "-" in rec_id else None
        if prefix:
            keys.append(f"{prefix}-{digits}")
            keys.append(f"{prefix}_{digits}")
            keys.append(f"{prefix}{digits}")
    # unique keep order
    out, seen = [], set()
    for k in keys:
        if k and k not in seen:
            out.append(k); seen.add(k)
    return out

# 5-class mapping: W=0, N1=1, N2=2, N3=3 (S3/S4 merged), REM=4, unknown=-1
def map_stage_to_5class(stage):
    if stage is None:
        return -1
    s = str(stage).strip().upper()

    if s in ["W", "WAKE", "0"]:
        return 0
    if s in ["N1", "S1", "STAGE1", "STAGE 1", "1"]:
        return 1
    if s in ["N2", "S2", "STAGE2", "STAGE 2", "2"]:
        return 2
    if s in ["N3", "S3", "S4", "STAGE3", "STAGE 3", "STAGE4", "STAGE 4", "3", "4"]:
        return 3
    if s in ["R", "REM", "5"]:
        return 4

    # unknown / movement
    if s in ["9", "-1", "?", "UNKNOWN", "MOVEMENT", "MT"]:
        return -1

    # fallback numeric
    try:
        v = int(float(s))
        return map_stage_to_5class(v)
    except:
        return -1

LABEL_NAMES = {0:"W",1:"N1",2:"N2",3:"N3",4:"REM",-1:"UNK"}
print("UTILS OK")


UTILS OK


In [4]:
# Cell 4 — Find Profusion XML for a record
def find_profusion_xml(rec_id: str, annot_root: Path):
    if not annot_root.exists():
        return None

    keys = normalize_keys_from_rec_id(rec_id)
    # Prefer *profusion.xml
    hits = []
    for p in annot_root.rglob("*.xml"):
        name = p.name.lower()
        if "profusion" not in name:
            continue
        if any(k.lower() in name for k in keys):
            hits.append(p)

    if not hits:
        return None
    # shortest name first
    hits.sort(key=lambda p: len(p.name))
    return hits[0]

print("Profusion finder ready.")


Profusion finder ready.


In [5]:
# Cell 5 — Profusion XML staging parser (SleepStage tokens)
import xml.etree.ElementTree as ET

def _strip_ns(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag

def parse_stages_from_profusion_xml(xml_path: Path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    stages = []
    for elem in root.iter():
        t = _strip_ns(elem.tag).lower()
        if "sleepstage" in t:
            if elem.text is not None:
                s = elem.text.strip()
                if s != "":
                    stages.append(s)

    if len(stages) < 10:
        # fallback: look for stage in attributes
        for elem in root.iter():
            for k, v in elem.attrib.items():
                if "stage" in k.lower():
                    stages.append(str(v).strip())

    if len(stages) < 10:
        raise ValueError(f"Too few stage tokens in {xml_path.name}: {len(stages)}")

    y = np.array([map_stage_to_5class(s) for s in stages], dtype=np.int64)
    return y

print("Profusion parser ready.")


Profusion parser ready.


In [6]:
# Cell 6 — EEG channel picker (EEG only = C4–A1 equiv)
def pick_eeg_channel_only(ch_names):
    if EEG_CHANNEL_NAME in ch_names:
        return EEG_CHANNEL_NAME, "C4A1_equiv_EEG"
    # fuzzy fallback: exact 'eeg' ignoring spaces/case
    for c in ch_names:
        if c.strip().lower() == "eeg":
            return c, "C4A1_equiv_EEG_fuzzy"
    return None, None

print("Channel picker ready (EEG only).")


Channel picker ready (EEG only).


In [7]:
# Cell 7 — Signal extraction: bandpass + resample + epoching + align labels
def load_single_channel_epochs(edf_path: Path, channel: str, y: np.ndarray,
                              target_fs=125, epoch_sec=30,
                              bp_low=0.5, bp_high=40.0):
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    raw.pick_channels([channel])

    # bandpass
    raw.filter(l_freq=bp_low, h_freq=bp_high, verbose=False)

    # resample
    fs = float(raw.info["sfreq"])
    if abs(fs - target_fs) > 1e-6:
        raw.resample(target_fs, npad="auto", verbose=False)
        fs = float(raw.info["sfreq"])

    sig = raw.get_data()[0].astype(np.float32)
    epoch_len = int(epoch_sec * fs)
    n_epochs_sig = len(sig) // epoch_len
    sig = sig[:n_epochs_sig * epoch_len]
    x = sig.reshape(n_epochs_sig, epoch_len)

    # align to labels
    n = min(len(y), n_epochs_sig)
    x = x[:n]
    y2 = y[:n].astype(np.int64)

    return x, y2, int(fs)

def trim_epochs(x, y, trim_start_min, trim_end_min, epoch_sec=30):
    trim_start = int((trim_start_min * 60) / epoch_sec)
    trim_end   = int((trim_end_min * 60) / epoch_sec)
    if len(y) <= (trim_start + trim_end + 1):
        return None, None
    return x[trim_start:len(x)-trim_end], y[trim_start:len(y)-trim_end]

def compute_tst_hours(y, epoch_sec=30):
    sleep_mask = np.isin(y, [1,2,3,4])  # N1,N2,N3,REM
    return float(sleep_mask.sum() * epoch_sec / 3600.0)

def robust_zscore_per_record(x):
    flat = x.reshape(-1)
    med = np.median(flat)
    iqr = np.percentile(flat, 75) - np.percentile(flat, 25)
    if iqr < 1e-6:
        mu = float(np.mean(flat))
        sd = float(np.std(flat) + 1e-6)
        return (x - mu) / sd
    return (x - med) / (iqr + 1e-6)

print("Epoching + trim + TST + normalize ready.")


Epoching + trim + TST + normalize ready.


In [8]:
# Cell 8 — One record → NPZ (Plan A rules)
def process_one_record(edf_path: Path, visit: str, out_dir: Path, dry_run=False):
    rec_id = infer_record_id_from_edf(edf_path)

    # annotation
    xml_path = find_profusion_xml(rec_id, ANNOT_DIR)
    if xml_path is None:
        return {"rec_id": rec_id, "status":"no_profusion_xml", "edf":str(edf_path), "visit":visit}

    try:
        y = parse_stages_from_profusion_xml(xml_path)
    except Exception as e:
        return {"rec_id": rec_id, "status":"profusion_parse_fail", "error":str(e), "edf":str(edf_path), "visit":visit, "xml":str(xml_path)}

    # channel
    raw_head = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
    ch, ch_tag = pick_eeg_channel_only(raw_head.ch_names)
    if ch is None:
        return {"rec_id": rec_id, "status":"no_EEG_channel", "edf":str(edf_path), "visit":visit, "xml":str(xml_path), "channels":raw_head.ch_names}

    # epochs
    try:
        x, y2, fs = load_single_channel_epochs(edf_path, ch, y, target_fs=TARGET_FS, epoch_sec=EPOCH_SEC)
    except Exception as e:
        return {"rec_id": rec_id, "status":"edf_read_fail", "error":str(e), "edf":str(edf_path), "visit":visit, "xml":str(xml_path), "channel":ch}

    # trim
    x, y2 = trim_epochs(x, y2, TRIM_MINUTES_START, TRIM_MINUTES_END, EPOCH_SEC)
    if x is None:
        return {"rec_id": rec_id, "status":"too_short_after_trim", "edf":str(edf_path), "visit":visit, "xml":str(xml_path), "channel":ch}

    # TST filter
    tst_h = compute_tst_hours(y2, EPOCH_SEC)
    if tst_h <= MIN_TST_HOURS:
        return {"rec_id": rec_id, "status":"tst_too_short", "tst_h":tst_h, "edf":str(edf_path), "visit":visit, "xml":str(xml_path), "channel":ch}

    # normalize
    x = robust_zscore_per_record(x).astype(np.float32)

    # save
    out_path = out_dir / f"{rec_id}_{visit}.npz"
    if not dry_run:
        np.savez_compressed(
            out_path,
            x=x,
            y=y2.astype(np.int64),
            fs=np.int32(fs),
            channel=str(ch),
            channel_tag=str(ch_tag),
            visit=str(visit),
            edf=str(edf_path),
            annot=str(xml_path),
            trim_start_min=np.int32(TRIM_MINUTES_START),
            trim_end_min=np.int32(TRIM_MINUTES_END),
        )

    return {
        "rec_id": rec_id,
        "status":"ok",
        "visit": visit,
        "npz": str(out_path),
        "fs": fs,
        "channel": ch,
        "epochs": int(len(y2)),
        "tst_h": float(tst_h),
    }

print("process_one_record ready.")


process_one_record ready.


In [9]:
# Cell 9 — STEP 1: Pilot EDF inspection (channels + duration)
edfs1 = list_edf_files(EDF_DIR_MESA)
print("MESA EDF count:", len(edfs1))
pilot_edfs = edfs1[:PILOT_N]

for p in pilot_edfs[:5]:
    raw = mne.io.read_raw_edf(p, preload=False, verbose=False)
    sf = float(raw.info["sfreq"])
    dur_h = raw.n_times / sf / 3600
    print("="*80)
    print(p.name, "| fs:", sf, "| dur(h):", round(dur_h,2), "| ch:", raw.ch_names)


MESA EDF count: 2056
mesa-sleep-0001.edf | fs: 256.0 | dur(h): 12.0 | ch: ['EKG', 'EOG-L', 'EOG-R', 'EMG', 'EEG1', 'EEG2', 'EEG3', 'Pres', 'Flow', 'Snore', 'Thor', 'Abdo', 'Leg', 'Therm', 'Pos', 'EKG_Off', 'EOG-L_Off', 'EOG-R_Off', 'EMG_Off', 'EEG1_Off', 'EEG2_Off', 'EEG3_Off', 'Pleth', 'OxStatus', 'SpO2', 'HR', 'DHR']
mesa-sleep-0002.edf | fs: 256.0 | dur(h): 11.0 | ch: ['EKG', 'EOG-L', 'EOG-R', 'EMG', 'EEG1', 'EEG2', 'EEG3', 'Pres', 'Flow', 'Snore', 'Thor', 'Abdo', 'Leg', 'Therm', 'Pos', 'EKG_Off', 'EOG-L_Off', 'EOG-R_Off', 'EMG_Off', 'EEG1_Off', 'EEG2_Off', 'EEG3_Off', 'Pleth', 'OxStatus', 'SpO2', 'HR', 'DHR']
mesa-sleep-0006.edf | fs: 256.0 | dur(h): 9.0 | ch: ['EKG', 'EOG-L', 'EOG-R', 'EMG', 'EEG1', 'EEG2', 'EEG3', 'Pres', 'Flow', 'Snore', 'Thor', 'Abdo', 'Leg', 'Therm', 'Pos', 'EKG_Off', 'EOG-L_Off', 'EOG-R_Off', 'EMG_Off', 'EEG1_Off', 'EEG2_Off', 'EEG3_Off', 'Pleth', 'OxStatus', 'SpO2', 'HR', 'DHR']
mesa-sleep-0010.edf | fs: 256.0 | dur(h): 10.0 | ch: ['EKG', 'EOG-L', 'EOG-R', '

In [10]:
# Cell 10 — STEP 2: Pilot annotation check (Profusion XML)
for p in pilot_edfs:
    rec_id = infer_record_id_from_edf(p)
    xml = find_profusion_xml(rec_id, ANNOT_DIR)
    if xml is None:
        print("[XML]", rec_id, "NOT FOUND")
        continue
    y = parse_stages_from_profusion_xml(xml)
    counts = dict(pd.Series(y).value_counts().sort_index())
    print("[XML]", rec_id, "|", xml.name, "| epochs:", len(y), "| counts:", counts)

[XML] mesa-sleep-0001 | mesa-sleep-0001-profusion.xml | epochs: 1439 | counts: {0: np.int64(752), 1: np.int64(135), 2: np.int64(455), 3: np.int64(19), 4: np.int64(78)}
[XML] mesa-sleep-0002 | mesa-sleep-0002-profusion.xml | epochs: 1319 | counts: {0: np.int64(571), 1: np.int64(49), 2: np.int64(362), 3: np.int64(156), 4: np.int64(181)}
[XML] mesa-sleep-0006 | mesa-sleep-0006-profusion.xml | epochs: 1079 | counts: {0: np.int64(364), 1: np.int64(141), 2: np.int64(380), 3: np.int64(75), 4: np.int64(119)}
[XML] mesa-sleep-0010 | mesa-sleep-0010-profusion.xml | epochs: 1199 | counts: {0: np.int64(980), 1: np.int64(20), 2: np.int64(126), 3: np.int64(73)}
[XML] mesa-sleep-0012 | mesa-sleep-0012-profusion.xml | epochs: 1427 | counts: {0: np.int64(882), 1: np.int64(85), 2: np.int64(376), 3: np.int64(16), 4: np.int64(68)}
[XML] mesa-sleep-0014 | mesa-sleep-0014-profusion.xml | epochs: 1679 | counts: {0: np.int64(836), 1: np.int64(22), 2: np.int64(381), 3: np.int64(262), 4: np.int64(178)}
[XML] me

In [11]:
# Cell 11 — STEP 3+4: Pilot NPZ creation (Plan A)
pilot_results = []
for p in pilot_edfs:
    res = process_one_record(p, visit="v1", out_dir=OUT_MESA, dry_run=False)
    pilot_results.append(res)
    print(res["rec_id"], "|", res["status"], "| TST:", res.get("tst_h"), "| npz:", res.get("npz"))

pilot_df = pd.DataFrame(pilot_results)
print("\nStatus counts:\n", pilot_df["status"].value_counts())
pilot_df

mesa-sleep-0001 | ok | TST: 5.725 | npz: mesa_sleepstaging_planA\npz_mesa\mesa-sleep-0001_v1.npz
mesa-sleep-0002 | ok | TST: 6.233333333333333 | npz: mesa_sleepstaging_planA\npz_mesa\mesa-sleep-0002_v1.npz
mesa-sleep-0006 | ok | TST: 5.958333333333333 | npz: mesa_sleepstaging_planA\npz_mesa\mesa-sleep-0006_v1.npz
mesa-sleep-0010 | tst_too_short | TST: 1.825 | npz: None
mesa-sleep-0012 | ok | TST: 4.541666666666667 | npz: mesa_sleepstaging_planA\npz_mesa\mesa-sleep-0012_v1.npz
mesa-sleep-0014 | ok | TST: 7.025 | npz: mesa_sleepstaging_planA\npz_mesa\mesa-sleep-0014_v1.npz
mesa-sleep-0016 | ok | TST: 6.9 | npz: mesa_sleepstaging_planA\npz_mesa\mesa-sleep-0016_v1.npz
mesa-sleep-0021 | ok | TST: 7.2 | npz: mesa_sleepstaging_planA\npz_mesa\mesa-sleep-0021_v1.npz
mesa-sleep-0027 | ok | TST: 8.2 | npz: mesa_sleepstaging_planA\npz_mesa\mesa-sleep-0027_v1.npz
mesa-sleep-0028 | ok | TST: 5.216666666666667 | npz: mesa_sleepstaging_planA\npz_mesa\mesa-sleep-0028_v1.npz

Status counts:
 status
ok  

,rec_id,status,visit,npz,fs,channel,epochs,tst_h,edf,xml
0,mesa-sleep-0001,ok,v1,mesa_sleepstaging_planA\npz_mesa\mesa-sleep-00...,125.0,EEG1,1359.0,5.725000,NaN,NaN
1,mesa-sleep-0002,ok,v1,mesa_sleepstaging_planA\npz_mesa\mesa-sleep-00...,125.0,EEG1,1239.0,6.233333,NaN,NaN
2,mesa-sleep-0006,ok,v1,mesa_sleepstaging_planA\npz_mesa\mesa-sleep-00...,125.0,EEG1,999.0,5.958333,NaN,NaN
3,mesa-sleep-0010,tst_too_short,v1,NaN,NaN,EEG1,NaN,1.825000,C:\Users\Zubair\mesa\polysomnography\edfs\mesa...,C:\Users\Zubair\mesa\polysomnography\annotatio...
4,mesa-sleep-0012,ok,v1,mesa_sleepstaging_planA\npz_mesa\mesa-sleep-00...,125.0,EEG1,1347.0,4.541667,NaN,NaN
5,mesa-sleep-0014,ok,v1,mesa_sleepstaging_planA\npz_mesa\mesa-sleep-00...,125.0,EEG1,1599.0,7.025000,NaN,NaN
6,mesa-sleep-0016,ok,v1,mesa_sleepstaging_planA\npz_mesa\mesa-sleep-00...,125.0,EEG1,1119.0,6.900000,NaN,NaN
7,mesa-sleep-0021,ok,v1,mesa_sleepstaging_planA\npz_mesa\mesa-sleep-00...,125.0,EEG1,999.0,7.200000,NaN,NaN
8,mesa-sleep-0027,ok,v1,mesa_sleepstaging_planA\npz_mesa\mesa-sleep-00...,125.0,EEG1,1119.0,8.200000,NaN,NaN
9,mesa-sleep-0028,ok,v1,mesa_sleepstaging_planA\npz_mesa\mesa-sleep-00...,125.0,EEG1,1059.0,5.216667,NaN,NaN


In [12]:
# Cell 12 — STEP 4: Inspect NPZ (one example + distribution)
ok = pilot_df[pilot_df["status"]=="ok"]
print("OK pilot records:", len(ok))

if len(ok) > 0:
    sample_npz = ok.iloc[0]["npz"]
    d = np.load(sample_npz, allow_pickle=True)
    x, y = d["x"], d["y"]
    print("\nSample:", sample_npz)
    print("x:", x.shape, x.dtype, "| y:", y.shape, y.dtype)
    print("fs:", int(d["fs"]), "| channel:", d["channel"], "| tag:", d["channel_tag"])
    print("TST(h) after trim:", compute_tst_hours(y, EPOCH_SEC))
    print("label_counts:", dict(pd.Series(y).value_counts().sort_index()))

OK pilot records: 9

Sample: mesa_sleepstaging_planA\npz_mesa\mesa-sleep-0001_v1.npz
x: (1359, 3750) float32 | y: (1359,) int64
fs: 125 | channel: EEG1 | tag: C4A1_equiv_EEG
TST(h) after trim: 5.725
label_counts: {0: np.int64(672), 1: np.int64(135), 2: np.int64(455), 3: np.int64(19), 4: np.int64(78)}


In [13]:
# Cell 13 — STEP 5a: Build manifests (EDF paths)
def build_manifest(edf_dir: Path, visit: str, out_csv: Path):
    edfs = list_edf_files(edf_dir)
    df = pd.DataFrame([{"rec_id": infer_record_id_from_edf(p), "edf": str(p), "visit": visit} for p in edfs])
    df.to_csv(out_csv, index=False)
    print("Wrote:", out_csv, "| n =", len(df))
    return df

manifest_mesa = build_manifest(EDF_DIR_MESA, "v1", MANIFEST_DIR / "manifest_mesa.csv")


Wrote: mesa_sleepstaging_planA\manifests\manifest_mesa.csv | n = 2056


In [14]:
# Cell — FULL SCALE with PROGRESS BAR (SHHS1 + SHHS2)

from tqdm import tqdm
import pandas as pd
from pathlib import Path

def convert_manifest_to_npz_with_pbar(
    manifest_df: pd.DataFrame,
    out_dir: Path,
    limit=None,
    skip_existing=True,
    desc="Converting"
):
    rows = manifest_df.to_dict("records")
    if limit is not None:
        rows = rows[:limit]

    # count how many will actually be processed
    todo = []
    for r in rows:
        rec_id, visit = r["rec_id"], r["visit"]
        out_path = out_dir / f"{rec_id}_{visit}.npz"
        if skip_existing and out_path.exists():
            continue
        todo.append(r)

    results = []
    for r in tqdm(todo, desc=desc, total=len(todo)):
        rec_id, visit = r["rec_id"], r["visit"]
        edf = Path(r["edf"])

        res = process_one_record(edf, visit=visit, out_dir=out_dir, dry_run=False)
        results.append(res)

        if res["status"] != "ok":
            tqdm.write(f"[WARN] {rec_id} {visit} -> {res['status']} {res.get('tst_h')}")

    df = pd.DataFrame(results)
    if len(df) > 0:
        print("\nStatus counts:")
        print(df["status"].value_counts())

    return df



In [15]:
# FULL MESA
full_mesa = convert_manifest_to_npz_with_pbar(
    manifest_mesa,
    OUT_MESA,
    limit=None,
    skip_existing=True,
    desc="MESA → NPZ"
)

print("\nFinal NPZ counts:")
print("MESA:", len(list(OUT_MESA.glob("*.npz"))))



MESA → NPZ:   0%|          | 1/2047 [00:54<30:49:35, 54.24s/it]

[WARN] mesa-sleep-0010 v1 -> tst_too_short 1.825


MESA → NPZ:   0%|          | 3/2047 [01:44<18:39:58, 32.88s/it]

[WARN] mesa-sleep-0035 v1 -> tst_too_short 3.683333333333333


MESA → NPZ:   0%|          | 4/2047 [02:18<18:55:38, 33.35s/it]

...

[WARN] mesa-sleep-4480 v1 -> tst_too_short 2.658333333333333


MESA → NPZ:  66%|██████▌   | 1342/2047 [32:13:26<10:49:19, 55.26s/it]

[WARN] mesa-sleep-4499 v1 -> tst_too_short 3.875


MESA → NPZ:  67%|██████▋   | 1375/2047 [33:04:20<16:09:48, 86.59s/it] 


KeyboardInterrupt: 